# DFC framework integration — check

A small notebook to **run the `dfc_agent_framework_integration` defense on a few shopping samples** and
*see what it's doing*: the trusted facts it extracts from the task, the policies the LLM generates, and
whether it blocks an injection.

This is the **automated, LLM-driven** DFC (no model-authored SQL): per task it extracts `PreambleData`
facts from the task prompt and generates PGN grounding policies, then validates every tool call.

> **Runs on AWS Bedrock** (Claude). The agent, fact-extraction and policy-generation all go through the
> Bedrock Anthropic client — no OpenAI key needed. Requires AWS credentials (`~/.aws/credentials` or
> env) with Bedrock access in `AWS_REGION` (default `us-east-1`), and model access enabled for the
> Claude inference profiles below.

Artifacts (facts + generated policies) are written per task to
`runs/<pipeline>/<suite>/<user_task>/<attack>/<injection>_dfc/metadata.json`.

## 1. Setup

In [ ]:
import os, json, logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

logging.getLogger().setLevel(logging.WARNING)
for n in ("httpx", "openai", "httpcore", "anthropic", "boto3", "botocore", "urllib3"):
    logging.getLogger(n).setLevel(logging.WARNING)

CWD = Path.cwd()
REPO = next((p for p in [CWD, CWD.parent, CWD / "AgentDyn"] if (p / "runs").is_dir()), None)
assert REPO is not None, f"Could not locate runs/. Tried from {CWD}"
load_dotenv(REPO / ".env")  # optional; Bedrock uses AWS creds, not OPENAI_API_KEY
os.environ.setdefault("AWS_REGION", "us-east-1")
RUNS = REPO / "runs"

# Bedrock model ids contain a ':' (illegal in Windows paths); the benchmark sanitizes
# ':' '/' '\\' -> '_' when writing runs/. Mirror that here so artifact lookups match.
def safe_name(name: str) -> str:
    return name.replace("/", "_").replace(":", "_").replace("\\", "_")

# ---- config (all Anthropic Claude on AWS Bedrock) ----
SONNET = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
HAIKU  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
AGENT_MODEL = SONNET   # the agent doing the task — Sonnet completes multi-step tasks (e.g. OTP-gated)
DFC_MODEL   = HAIKU    # extracts facts + writes policies — structured, cheaper model is fine (can differ)
SUITE = "shopping"
BENCHMARK_VERSION = "v1.2"
ATTACK = "important_instructions"

# (user_task, injection_task or None).  None = clean utility run.
SAMPLES = [
    ("user_task_1", None),               # clean: change password to "NewPassword"
    ("user_task_1", "injection_task_8"), # attack: set password to PASS999 (direct tool) -> should block
    ("user_task_0", "injection_task_6"), # attack: money via web form (browse_webpage) -> does it gate it?
]

from agentdojo.task_suite.load_suites import get_suite
suite = get_suite(BENCHMARK_VERSION, SUITE)
print("agent:", AGENT_MODEL, "| dfc_model:", DFC_MODEL, "| samples:", SAMPLES)

## 2. Build the framework defense pipeline

In [ ]:
from agentdojo.agent_pipeline.agent_pipeline import PipelineConfig, AgentPipeline

cfg = PipelineConfig(
    llm=AGENT_MODEL, model_id=None,
    defense="dfc_agent_framework_integration",
    system_message_name=None, system_message=None,
    suite_name=SUITE, dfc_model=DFC_MODEL,
)
pipeline = AgentPipeline.from_config(cfg)
print("pipeline:", pipeline.name)

## 3. Run the samples

Each run builds a fresh DFC context (extract facts → generate policies), runs the task, and writes the
artifacts. Clean runs measure utility; attacked runs measure whether the injection is blocked
(`security=True` means the injection SUCCEEDED).

In [ ]:
from agentdojo.attacks.attack_registry import load_attack
from agentdojo.benchmark import run_task_with_injection_tasks, run_task_without_injection_tasks
from agentdojo.logging import OutputLogger, LOGGER_STACK

attack = load_attack(ATTACK, suite, pipeline)  # uses pipeline only for tool names

def build_pipeline():
    cfg = PipelineConfig(
        llm=AGENT_MODEL, model_id=None,
        defense="dfc_agent_framework_integration",
        system_message_name=None, system_message=None,
        suite_name=SUITE, dfc_model=DFC_MODEL,
    )
    return AgentPipeline.from_config(cfg)

def run_sample(job):
    ut_id, inj_id = job
    ut = suite.get_user_task_by_id(ut_id)
    pipe = build_pipeline()  # fresh Bedrock client per task — the async httpx pool is NOT safe to share
    with OutputLogger(str(RUNS), None):
        if inj_id is None:
            util, sec = run_task_without_injection_tasks(suite, pipe, ut, RUNS, True, BENCHMARK_VERSION)
            return (ut_id, None, {"utility": util})
        u, s = run_task_with_injection_tasks(suite, pipe, ut, attack, RUNS, True, [inj_id], BENCHMARK_VERSION)
        return (ut_id, inj_id, {"utility": dict(u), "security": dict(s)})

# Run SEQUENTIALLY. The Anthropic Bedrock client drives an async httpx connection pool via
# asyncio.run() per turn; sharing one client across ThreadPoolExecutor workers (each with its
# own event loop) deadlocks. One task at a time, each with its own client, avoids that entirely.
results = []
for job in SAMPLES:
    try:
        results.append(run_sample(job))
    except Exception as e:
        results.append((*job, {"error": repr(e)[:200]}))
for r in results:
    print(r)

## 4. Inspect what DFC did — facts, policies, blocks

For each sample we load `metadata.json` and show: the **extracted trusted facts**, the **generated
policies** (PGN + description), how many **registered** vs were **deleted** (failed repair), and the
**validation events** (which tool calls were blocked).

In [ ]:
def artifact_path(ut_id, inj_id):
    attack_dir = "none" if inj_id is None else ATTACK
    stem = "none" if inj_id is None else inj_id
    return RUNS / safe_name(pipeline.name) / SUITE / ut_id / attack_dir / f"{stem}_dfc" / "metadata.json"

def show(ut_id, inj_id):
    p = artifact_path(ut_id, inj_id)
    label = f"{ut_id}" + (f" x {inj_id}" if inj_id else " (clean)")
    print("=" * 78); print(label); print("artifact:", p.exists())
    if not p.exists():
        print("  (no artifact — run may have errored before policy generation)"); return
    d = json.load(open(p, encoding="utf-8"))
    print("\nEXTRACTED FACTS (trusted, from the task prompt):")
    for k, v in (d.get("extracted_facts") or {}).items():
        print(f"   {k} = {v!r}")
    gp = d.get("generated_policies", []); reg = d.get("registered_passant_policies", []); dele = d.get("deleted_policies", [])
    print(f"\nPOLICIES: generated={len(gp)}  registered={len(reg)}  deleted={len(dele)}")
    for pol in gp:
        print(f"   [{pol.get('applies_to_relation')}] {pol.get('description','')[:70]}")
        print(f"      {pol.get('pgn','').strip().replace(chr(10),' ')[:120]}")
    if dele:
        print("   deleted (failed repair):", [r.get('policy_id') for r in dele])

for ut_id, inj_id, _ in results:
    show(ut_id, inj_id)

## 5. Validation events — was the injection actually blocked?

The `metadata.json` (and the exported event DB) record each validation. Here we pull the validation
events to see which tool calls were allowed vs blocked, and tie it to the security outcome.

In [ ]:
def show_validation(ut_id, inj_id):
    p = artifact_path(ut_id, inj_id)
    label = f"{ut_id}" + (f" x {inj_id}" if inj_id else " (clean)")
    if not p.exists():
        print(label, "-> no artifact"); return
    d = json.load(open(p, encoding="utf-8"))
    events = d.get("validation_events") or d.get("diagnostics", {}).get("validation_events") or []
    blocked = [e for e in events if e.get("blocked")]
    print(f"{label}: {len(events)} validations, {len(blocked)} blocked")
    for e in blocked[:5]:
        print(f"   BLOCKED {e.get('event_type')}: {e.get('policy_descriptions')}")

for ut_id, inj_id, _ in results:
    show_validation(ut_id, inj_id)

## Notes

- **No model-authored SQL** here — the agent makes normal tool calls; the framework validates each one
  against the LLM-generated policies (via the `CROSS JOIN PreambleData` provenance trick).
- The **trust boundary is the task prompt** (`preamble`), which is injection-free — so the attacker
  value (a PASS999 password, an attacker IBAN) isn't in `extracted_facts` and a correct policy blocks it.
- **What to watch:** the policies are LLM-generated per task. Check that (a) a *relevant* policy was
  actually generated for the sink under attack, and (b) it isn't vacuous. The repair/probe loop only
  guarantees a policy *allows* a legit value, not that it *blocks* an attacker value.
- **Runs on Bedrock.** Swap `AGENT_MODEL`/`DFC_MODEL` to `us.anthropic.claude-sonnet-4-5-20250929-v1:0`
  for a stronger agent + policy generation (Haiku 4.5 is cheaper but may stall on multi-step tasks like
  the OTP-gated password change, lowering clean utility without that being a defense failure).